# Analyze and plot results for networks

This script takes in the results from run_over_regs.sh, assembles them into a single dataset, and performs analyses on the data.

In [6]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

import matplotlib as mpl
import matplotlib.ticker as ticker

import numpy as np
import pandas as pd
from pydmd import DMD
import os
import plotly.graph_objects as go
import plotly.express as px

# import clustering packages
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
import seaborn as sns
from celluloid import Camera

plt.style.use('custom.mplstyle')
%config InlineBackend.figure_format = 'retina'
from tqdm import tqdm

from stoch_sim_model import *

## 0. Load data and build datasets

In [7]:
# Load data from infections
reg_model = ''
runs = '-1-'
comment = "prim-Nact-Ediv-vir" #"full-reg-vir" #"act-reg-exp-reg" # mem-reg

d_mean = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/raw/stacked_data'+runs+'runs'+'-'+comment+'.pkl'
mean_df = pd.read_pickle(d_mean)

with pd.option_context('display.max_columns', None):
    display(mean_df)

,psi_Nact_I,psi_Nact_H,psi_Nact_IH,F0_Nact,psi_NM_I,psi_NM_H,psi_NM_IH,F0_NM,psi_EM_I,psi_EM_H,psi_EM_IH,F0_EM,psi_Ediv_I,psi_Ediv_H,psi_Ediv_IH,F0_Ediv,d_I,K_IE,b_I,S_0,I_0,d_S,d_IE,d_IH,K_IH,A_init,b_Ain,b_H,d_H,K_EI,K_EH,N_0,max_Na,b_myc,d_myc,myc_thresh,t_act,t_bind,t_Na_div,t_E_div,t_M_div,t_E_die,t_E_cyt,p_load,s_load,T_max_pI,T_min_pI,harm_pI,harm_sI,harm_pS,harm_sS,max_pE,max_sE,T_pE,T_sE,inf_pM,inf_sM,init_M,int_pE,int_sE,int_pH,int_sH,min_pS,min_sS
0,-2.0,-2.0,0.0,-2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,-2.0,0.0,-2.0,0.1,10000.0,7.500000e-08,10000000.0,1000.0,0.01,12.0,0.0,5000.0,1000.0,1.0,1.0,2.0,10000.0,5000.0,100.0,8.0,1592.428682,2.376505,398.107171,1.0,0.75,0.25,0.333333,0.5,2.5,0.666667,6.731708e+07,6.731708e+07,17.552,0.000,1.073153e+07,1.073153e+07,0.000000e+00,0.000000e+00,0.0,0.0,0.000,0.000,0.0,0.0,0.0,0.000,0.000,3.261949e+06,3.261949e+06,3.288419e+05,3.288419e+05
1,-2.0,-2.0,0.0,-2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,-2.0,0.0,-2.0,0.1,10000.0,8.750000e-08,10000000.0,1000.0,0.01,12.0,0.0,5000.0,1000.0,1.0,1.0,2.0,10000.0,5000.0,100.0,8.0,1592.428682,2.376505,398.107171,1.0,0.75,0.25,0.333333,0.5,2.5,0.666667,7.639351e+07,7.639351e+07,14.944,0.000,1.100231e+07,1.100231e+07,0.000000e+00,0.000000e+00,0.0,0.0,0.000,0.000,0.0,0.0,0.0,0.000,0.000,3.732283e+06,3.732283e+06,2.553477e+05,2.553477e+05
2,-2.0,-2.0,0.0,-2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,-2.0,0.0,-2.0,0.1,10000.0,1.000000e-07,10000000.0,1000.0,0.01,12.0,0.0,5000.0,1000.0,1.0,1.0,2.0,10000.0,5000.0,100.0,8.0,1592.428682,2.376505,398.107171,1.0,0.75,0.25,0.333333,0.5,2.5,0.666667,8.205078e+07,8.205078e+07,13.032,0.000,1.117694e+07,1.117694e+07,0.000000e+00,0.000000e+00,0.0,0.0,0.000,0.000,0.0,0.0,0.0,0.000,0.000,4.025399e+06,4.025399e+06,2.066293e+05,2.066293e+05
3,-2.0,-2.0,0.0,-2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,-2.0,0.0,-2.0,0.1,10000.0,1.125000e-07,10000000.0,1000.0,0.01,12.0,0.0,5000.0,1000.0,1.0,1.0,2.0,10000.0,5000.0,100.0,8.0,1592.428682,2.376505,398.107171,1.0,0.75,0.25,0.333333,0.5,2.5,0.666667,8.590094e+07,8.590094e+07,11.564,0.000,1.130867e+07,1.130867e+07,0.000000e+00,0.000000e+00,0.0,0.0,0.000,0.000,0.0,0.0,0.0,0.000,0.000,4.224572e+06,4.224572e+06,1.723286e+05,1.723286e+05
4,-2.0,-2.0,0.0,-2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,-2.0,0.0,-2.0,0.1,10000.0,1.250000e-07,10000000.0,1000.0,0.01,12.0,0.0,5000.0,1000.0,1.0,1.0,2.0,10000.0,5000.0,100.0,8.0,1592.428682,2.376505,398.107171,1.0,0.75,0.25,0.333333,0.5,2.5,0.666667,8.869707e+07,8.869707e+07,10.404,0.000,1.141441e+07,1.141441e+07,0.000000e+00,0.000000e+00,0.0,0.0,0.000,0.000,0.0,0.0,0.0,0.000,0.000,4.368962e+06,4.368962e+06,1.470959e+05,1.470959e+05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8778120,2.0,2.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,2.0,0.0,2.0,0.5,500000.0,7.500000e-08,10000000.0,1000.0,0.01,12.0,0.0,5000.0,1000.0,1.0,1.0,2.0,500000.0,5000.0,100.0,8.0,1592.428682,2.376505,398.107171,1.0,0.75,0.25,0.333333,0.5,2.5,0.666667,5.581065e+03,5.581065e+03,2.640,3.672,4.943170e+03,4.943170e+03,6.938799e+06,6.938799e+06,622448.0,622448.0,1.132,1.132,913.0,913.0,39.0,1593498.976,1593498.976,2.470963e+03,2.470963e+03,3.316737e+06,3.316737e+06
8778121,2.0,2.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,2.0,0.0,2.0,0.5,500000.0,8.750000e-08,10000000.0,1000.0,0.01,12.0,0.0,5000.0,1000.0,1.0,1.0,2.0,500000.0,5000.0,100.0,8.0,1592.428682,2.376505,398.107171,1.0,0.75,0.25,0.333333,0.5,2.5,0.666667,7.171096e+03,7.171096e+03,2.836,3.856,7.054079e+03,7.054079e+03,7.568940e+06,7.568940e+06,715724.0,715724.0,1.076,1.076,1010.0,1010.0,38.0,1838163.880,1838163.880,3.526427e+03,3.526427e+03,2.791301e+06,2.791301e+06
8778122,2.0,2.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,2.0,0.0,2.0,0.5,500000.0,1.000000e-07,10000000.0,1000.0,0.01,12.0,0.0,5000.0,1000.0,1.

## 1. Understanding the statistics of responses to an infection

In [8]:
# Create additional variables
K_IEs = np.unique(mean_df['K_IE'])
d_Is = np.unique(mean_df['d_I'])
b_S = np.mean(mean_df['d_S']*mean_df['S_0'])
N_0 = np.mean(mean_df['N_0'])
virs = np.unique(mean_df[['I_0','d_I','K_IE','b_I']].to_numpy(), axis = 0)

module_reg = {'Nact': Nact_reg, 'NM': NM_reg, 'EM': EM_reg, 'Ediv': Ediv_reg}
module_reg_labels = {'Nact': param_names[-16:-12], 'NM': param_names[-12:-8], 'EM': param_names[-8:-4], 'Ediv': param_names[-4:]}

if "full-reg" in comment:
    reg = Nact_reg + NM_reg + EM_reg + Ediv_reg
    reg_label = param_names[-len(reg):]
    modules = ['Nact', 'NM', 'EM', 'Ediv']
    reg_and = [Nact_reg[2]] + [NM_reg[2]] + [EM_reg[2]] + [Ediv_reg[2]]
    reg_bias = [Nact_reg[3]] + [NM_reg[3]] + [EM_reg[3]] + [Ediv_reg[3]]
else:
    reg = (Nact_reg if "Nact" in comment else []) + (NM_reg if "NM" in comment else []) + (EM_reg if "EM" in comment else []) + (Ediv_reg if "Ediv" in comment else [])
    reg_label = (param_names[-16:-12] if "Nact" in comment else []) + (param_names[-12:-8] if "NM" in comment else []) + (param_names[-8:-4] if "EM" in comment else []) + (param_names[-4:] if "Ediv" in comment else [])
    modules = (["Nact"] if "Nact" in comment else []) + (["NM"] if "NM" in comment else []) + (["EM"] if "EM" in comment else []) + (["Ediv"] if "Ediv" in comment else [])
    reg_or = ([Nact_reg[0:2]] if "Nact" in comment else []) + ([NM_reg[0:2]] if "NM" in comment else []) + ([EM_reg[0:2]] if "EM" in comment else []) + ([Ediv_reg[0:2]] if "Ediv" in comment else [])
    reg_and = ([Nact_reg[2]] if "Nact" in comment else []) + ([NM_reg[2]] if "NM" in comment else []) + ([EM_reg[2]] if "EM" in comment else []) + ([Ediv_reg[2]] if "Ediv" in comment else [])
    reg_and_label = ([param_names[-14]] if "Nact" in comment else []) + ([param_names[-10]] if "NM" in comment else []) + ([param_names[-6]] if "EM" in comment else []) + ([param_names[-2]] if "Ediv" in comment else [])
    reg_bias = ([Nact_reg[3]] if "Nact" in comment else []) + ([NM_reg[3]] if "NM" in comment else []) + ([EM_reg[3]] if "EM" in comment else []) + ([Ediv_reg[3]] if "Ediv" in comment else [])
    reg_bias_label = ([param_names[-13]] if "Nact" in comment else []) + ([param_names[-9]] if "NM" in comment else []) + ([param_names[-5]] if "EM" in comment else []) + ([param_names[-1]] if "Ediv" in comment else [])

mean_df['int_presp'] = mean_df['int_pE'] + mean_df['inf_pM']
mean_df['int_sresp'] = mean_df['int_sE'] + mean_df['inf_sM']

# identify Biologically evidenced networks
mean_df['bio_reg'] = (mean_df[EM_reg[0]] > 0.0)*(mean_df[EM_reg[1]] > 0.0)*(mean_df[EM_reg[2]] > 0.0)*1
keep_vars = ['harm_pI','T_min_pI', 'harm_pS', "init_M", "T_max_pI", "T_pE", "int_pE"]

In [10]:
# save data sets
cutoff = 1 - 0.01 if 0.01*mean_df.shape[0]/len(virs) > 1 else 1 - 10/(mean_df.shape[0]/len(virs))
infection_scenarios = []
no_eff_data = [[] for i in np.arange(len(virs))]

for l, (I_0, d_I, K_IE, b_I) in enumerate(virs):
    data = mean_df.loc[(mean_df["d_I"] == d_I)*(mean_df["K_IE"] == K_IE)*(mean_df["b_I"] == b_I), ['b_I','d_I', 'K_IE', 'I_0','S_0', 'N_0', 'd_S', 'K_EH'] + Nact_reg + NM_reg + EM_reg + Ediv_reg + keep_vars]

    # compute infection harm without T cell response
    no_eff_data[l] = lin_stoch_sim(N_0 = 0, I_0 = I_0, K_IE = K_IE, d_I = d_I, b_I = b_I)
    no_eff_stats = no_eff_data[l]["summary_stats"]
    
    data.loc[:,"peff_protection"] = (no_eff_stats[4] - data['harm_pI'].to_numpy())/(b_S*(data['T_min_pI'] + sim_duration*(data['T_min_pI'] == 0)))
    data.loc[:,"p_potential_harm"] = (1 - np.exp(1)*I_0/K_IE)*(1 - np.exp(1)*(d_I*I_0)/(b_I*S_0 - d_I)/K_EH) #*no_eff_stats[4]/(b_S*sim_duration)
    data.loc[:,"peff_toxicity"] = data['harm_pS'].to_numpy()/(b_S*(data['T_min_pI'] + sim_duration*(data['T_min_pI'] == 0)))
    data.loc[:,"peff_utility"] = (no_eff_stats[4] + 1.0*no_eff_stats[6] - (data['harm_pI'] + data['harm_pS']).to_numpy())/(b_S*(data['T_min_pI'] + sim_duration*(data['T_min_pI'] == 0)))

    if 'comp_model' not in comment:
        data.loc[:, "high_putility"] = 1*(data['peff_utility'] >= np.quantile(data['peff_utility'], cutoff))
        data.loc[:, "high_pprotection"] = 1*(data['peff_protection'] >= np.quantile(data['peff_protection'], cutoff))
        data.loc[:, "low_ptoxicity"] = 1*(-data['peff_toxicity'] >= np.quantile(-data['peff_toxicity'], cutoff))
        data.loc[:, "high_pmemory"] = 1*(data['init_M'] >= np.quantile(data['init_M'], cutoff))
        data.loc[:, "low_clear_timing"] = 1*(-data['T_max_pI'] >= np.quantile(-data['T_max_pI'], cutoff))
        data.loc[:, "low_resp_timing"] = 1*(-data['T_pE'] >= np.quantile(-data['T_pE'], cutoff))
        data.loc[:, "high_presponse"] = 1*(data['int_pE'] >= np.quantile(data['int_pE'], cutoff))

    infection_scenarios.append(data)

# stack datasets
clustered_mean_df = pd.concat(infection_scenarios)

clustered_mean_df.to_pickle('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/mean/processed_data'+runs+'runs'+'-'+comment+'.pkl')

/opt/minimamba/envs/maximmune/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3464: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/minimamba/envs/maximmune/lib/python3.11/site-packages/numpy/core/_methods.py:192: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/opt/minimamba/envs/maximmune/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3464: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/minimamba/envs/maximmune/lib/python3.11/site-packages/numpy/core/_methods.py:192: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/opt/minimamba/envs/maximmune/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3464: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/minimamba/envs/maximmune/lib/python3.11/site-packages/numpy/core/_methods.py:192: RuntimeWar